# 1.3 Memory

`agent.invoke` is **stateless by default**: each call is a new conversation unless you send previous messages yourself (as in the foundational-models notebook).

A **checkpointer** stores graph state between turns. You key that store with a `thread_id` so multiple chats can stay separate.

This notebook first shows a failed follow-up (no memory), then the same two turns with `InMemorySaver`.

## Setup

Load API keys from `.env`. Run this cell first.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## No memory

Create an agent **without** a checkpointer. Tell it your name and favourite colour, then in a **second** `invoke` ask for the colour — without passing the first messages back. The model has no record of the first turn.

In [ ]:
from langchain.agents import create_agent

# No checkpointer → nothing is saved between invoke() calls
agent = create_agent(
    "gpt-5-nano"
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)

In [ ]:
from pprint import pprint

# This turn includes the intro; the next invoke will not see it
pprint(response)

In [ ]:
question = HumanMessage(content="What's my favourite colour?")

# New invoke, only this one HumanMessage — the agent cannot know "green"
response = agent.invoke(
    {"messages": [question]} 
)

pprint(response)

## Memory with a checkpointer

`InMemorySaver` keeps state in process RAM (fine for notebooks; use a DB-backed saver in production).

Pass `config={"configurable": {"thread_id": "1"}}` on **every** `invoke` that belongs to the same chat. The same `thread_id` loads prior messages; a different id is a different conversation.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    checkpointer=InMemorySaver(),  # persist state keyed by thread_id
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")
# Thread id groups turns into one conversation
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [ ]:
pprint(response)

Same `config` / `thread_id`. You only send the new user message; the checkpointer prepends history. The agent should answer **green**.

In [ ]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  # same thread as the intro
)

pprint(response)